In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

# 1) تحميل التقييمات والأفلام
if os.path.exists("u.data"):
    ratings = pd.read_csv("u.data", sep="\t",
                          names=["user_id", "movie_id", "rating", "timestamp"])
    movies = pd.read_csv("u.item", sep="|", encoding="latin-1", header=None,
                         usecols=[0, 1], names=["movie_id", "title"])
else:
    ratings = pd.read_csv("ratings.csv").rename(
        columns={"userId": "user_id", "movieId": "movie_id"})
    movies = pd.read_csv("movies.csv").rename(columns={"movieId": "movie_id"})

titles = movies.set_index("movie_id")["title"]
print(ratings.shape, ratings["user_id"].nunique(), "users",
      ratings["movie_id"].nunique(), "movies")

# 2) تقسيم التقييمات: 80% تدريب و20% اختبار
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# 3) مصفوفة user-item
R = train.pivot_table(index="user_id", columns="movie_id", values="rating").fillna(0)

# 4) تشابه المستخدمين (cosine similarity)
sim_df = pd.DataFrame(cosine_similarity(R), index=R.index, columns=R.index)

# 5) دالة التوصية
def recommend_ids(user_id, k=10, n_neighbors=30, min_neighbors=3):
    sims = sim_df[user_id].drop(user_id).nlargest(n_neighbors)
    neighbors = R.loc[sims.index]
    weighted_sum = neighbors.T.dot(sims)
    weight_total = neighbors.gt(0).T.dot(sims) + 1e-9
    scores = weighted_sum / weight_total
    scores = scores[neighbors.gt(0).sum() >= min_neighbors]   # أفلام قيّمها كذا جار
    seen = R.loc[user_id]
    scores = scores.drop(seen[seen > 0].index, errors="ignore")  # نشيل اللي شافه
    scores = pd.to_numeric(scores, errors="coerce").dropna()
    return scores.nlargest(k).index

def recommend(user_id, k=10):
    return titles.reindex(recommend_ids(user_id, k))

print("Top 10 recommendations for user 1:")
print(recommend(1).to_string())

# 6) التقييم بـ Precision@K
relevant = test[test["rating"] >= 4].groupby("user_id")["movie_id"].apply(set)

def precision_at_k(k=10, n_users=300):
    scores = []
    for u in relevant.index[:n_users]:
        if u not in R.index:
            continue
        recs = set(recommend_ids(u, k))
        scores.append(len(recs & relevant[u]) / k)
    return np.mean(scores)

print("Precision@10:", round(precision_at_k(10), 3))

(100000, 4) 943 users 1682 movies
Top 10 recommendations for user 1:
movie_id
313                                Titanic (1997)
285                         Secrets & Lies (1996)
1007                   Waiting for Guffman (1996)
1019    Die xue shuang xiong (Killer, The) (1989)
661                              High Noon (1952)
221                     Breaking the Waves (1996)
64               Shawshank Redemption, The (1994)
963       Some Folks Call It a Sling Blade (1993)
408                         Close Shave, A (1995)
856                         Night on Earth (1991)
Precision@10: 0.083
